In [ ]:
from pathlib import Path
import yaml
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mdx2.io import loadobj
from mdx2.report.map_statistics import prepare_table, calc_intensity_statistics

# load the site config
with open("site_config.yaml") as f:
    config = yaml.safe_load(f)

# project directory
project_directory = Path(config['project_directory'])

In [ ]:
# calculate isotropic part

def load_merged_calc_stats(dset_root):

    crystal = loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'crystal')
    symmetry =  loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'symmetry')
    hkl_table =  loadobj(project_directory / dset_root / 'mac1_merge/mdx2/merged.nxs', 'hkl_table')

    df = prepare_table(hkl_table, crystal, symmetry)

    df_stats_excluded = calc_intensity_statistics(df, exclude_bragg=True, bin_width=0.01)
    df_stats = calc_intensity_statistics(df, exclude_bragg=False, bin_width=0.01)
    
    return df_stats, df_stats_excluded

data = {k:load_merged_calc_stats(k) for k in ['chess_20251105','chess_20250924','als_20251015','als_20251125']}


In [ ]:
plt.figure(figsize=(6,4))
for k, (df_stats, df_stats_excluded) in data.items():
    normval = np.max(df_stats_excluded['intensity_iso'])
    plt.plot(df_stats_excluded['s'], df_stats_excluded['intensity_iso'] / normval, label=k)
plt.xlabel("1/d (Å$^{-1}$)")
plt.ylabel("Intensity (normalized)")
plt.legend()
plt.ylim(0,1.05)
plt.xlim(0,0.8)
plt.title('Isotropic component')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
for k, (df_stats, df_stats_excluded) in data.items():
    normval = np.max(df_stats_excluded['intensity_iso'])
    s = df_stats['s'].array
    ivar = df_stats['intensity_sub_rms'].array**2
    evar = df_stats['intensity_error_rms'].array**2
    signal_var_est = ivar - evar
    signal_var_est[signal_var_est<0] = np.nan # suppress error sqrt(x) when x < 0
    irmscorr = np.sqrt(signal_var_est)
    plt.plot(s, irmscorr / normval, label=k)

plt.xlabel("1/d (Å$^{-1}$)")
plt.ylabel("RMS Intensity\n(normalized, corrected)")
plt.ylim(0,.125)
plt.xlim(0,0.8)
plt.legend()
plt.title('Anisotropic component (all voxels)')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
for k, (df_stats, df_stats_excluded) in data.items():
    normval = np.max(df_stats_excluded['intensity_iso'])
    s = df_stats['s'].array
    ivar = df_stats_excluded['intensity_sub_rms'].array**2
    evar = df_stats_excluded['intensity_error_rms'].array**2
    signal_var_est = ivar - evar
    signal_var_est[signal_var_est<0] = np.nan # suppress error sqrt(x) when x < 0
    irmscorr = np.sqrt(signal_var_est)
    plt.plot(s, irmscorr / normval, label=k)

plt.xlabel("1/d (Å$^{-1}$)")
plt.ylabel("R.M.S. Intensity\n(normalized, corrected)")
plt.ylim(0,0.07)
plt.xlim(0,0.8)
plt.legend()
plt.title('Anisotropic component (Bragg voxels excluded)')
plt.show()

In [ ]:
# Dump the curves into csv files for reuse in manuscript figure
for k, (df_stats, df_stats_excluded) in data.items():
    f = df_stats_excluded[['s','intensity_iso','intensity_sub_rms','intensity_error_rms']]
    f = f[(f['s']<=0.81) & (f['s']>0.03)] # trim resolution range
    normval = f['intensity_iso'].max()
    f['intensity_iso']/=normval
    f['intensity_sub_rms']/=normval
    f['intensity_error_rms']/=normval
    ivar = f['intensity_sub_rms'].array**2
    evar = f['intensity_error_rms'].array**2
    signal_var_est = ivar - evar
    signal_var_est[signal_var_est<0] = np.nan # suppress error sqrt(x) when x < 0
    irmscorr = np.sqrt(signal_var_est)
    f['intensity_aniso'] = np.sqrt(signal_var_est)
    f = f.set_index('s');
    f = f[['intensity_iso','intensity_aniso']]
    fn = Path('figures/source_data')/f'fig2_iso_aniso_{k}.csv'
    print('Writing',fn)
    f.to_csv(fn)

### Scrape crystal properties from DIALS logs

In [ ]:
def scrape_integrate_table(log_root="logs"):
    pats = {
        "Sigma_m (deg.)": r"sigma m:\s*([0-9.]+)",
        "Unit Cell": r"Unit cell:\s*(.+)",
    }
    rows = []
    for p in Path(log_root).glob("*/*/dials/dials.integrate.log"):
        t, ds, run = p.read_text(errors="ignore"), p.parts[-4], p.parts[-3]
        row = {"beamtime": ds, "crystal": run}
        for k, pat in pats.items():
            m = re.search(pat, t, re.M)
            row[k] = m.group(1) if m else None
        rows.append(row)
        # fix the unit cell cloumn
        if row["Unit Cell"] is not None:
            row["Unit Cell"] = tuple(
                float(re.sub(r"\(.*?\)", "", x).strip())
                for x in row["Unit Cell"].split(",")
            )
    return pd.DataFrame(rows).set_index(["beamtime", "crystal"]).sort_index()

integrate_df = scrape_integrate_table("logs")
integrate_df

In [ ]:
def scrape_merge_table(log_root="logs"):
    scalar_pats = {
        "Wilson B (Å^2)": r"Wilson B factor\s+([0-9.]+)",
        "Unit Cell": r"Using median unit cell across experiments\s*:\s*\(([^)]+)\)",
    }
    triplet_pats = {
        "I/sigma": r"^I/sigma\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)",
        "Multiplicity": r"^Multiplicity\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)",
    }
    labels = ("overall", "low", "high")
    rows = []

    for p in Path(log_root).glob("*/*/dials/dials.merge.log"):
        t, ds, run = p.read_text(errors="ignore"), p.parts[-4], p.parts[-3]
        row = {"beamtime": ds, "dataset": run}

        for k, pat in scalar_pats.items():
            m = re.search(pat, t, re.M)
            row[(k, "")] = m.group(1) if m else None

        if row[("Unit Cell", "")] is not None:
            row[("Unit Cell", "")] = tuple(float(x.strip()) for x in row[("Unit Cell", "")].split(","))

        if row[("Wilson B (Å^2)", "")] is not None:
            row[("Wilson B (Å^2)", "")] = float(row[("Wilson B (Å^2)", "")])

        for k, pat in triplet_pats.items():
            m = re.search(pat, t, re.M)
            vals = [float(x) for x in m.groups()] if m else [None, None, None]
            for lab, v in zip(labels, vals):
                row[(k, lab)] = v

        rows.append(row)

    df = pd.DataFrame(rows).set_index(["beamtime", "dataset"]).sort_index()
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df
    
merge_df = scrape_merge_table("logs")
merge_df